**The Environment Setup**

In [2]:
pip install ultralytics opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import time
from ultralytics import YOLO

# --- STEP 1: LOAD THE BRAIN ---
# Using YOLOv11 Nano for real-time speed on a laptop
model = YOLO('yolo11n.pt') 

# COCO Dataset IDs
PHONE_ID = 67
BOTTLE_ID = 39
CUP_ID = 41

# --- STEP 2: STATE INITIALIZATION ---
last_drink_time = time.time()
phone_start_time = None
HYDRATION_LIMIT = 30  # Demo threshold
DISTRACTION_LIMIT = 2

# --- STEP 3: START THE CAMERA ---
cap = cv2.VideoCapture(0)

print("AuraGuard Active. Focus on the popup window. Press 'q' to stop.")

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        current_time = time.time()
        
        # Inference - stream=True optimizes memory for video
        results = model(frame, stream=True, verbose=False)
        
        phone_present = False
        bottle_present = False

        # --- STEP 4: FILTERING & DETECTION ---
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                
                if cls == PHONE_ID and conf > 0.5:
                    phone_present = True
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    
                if cls in [BOTTLE_ID, CUP_ID] and conf > 0.5:
                    bottle_present = True
                    last_drink_time = current_time # Reset timer
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

        # --- STEP 5: TEMPORAL LOGIC (HUD) ---
        # Phone logic
        if phone_present:
            if phone_start_time is None: phone_start_time = current_time
            if (current_time - phone_start_time) > DISTRACTION_LIMIT:
                cv2.putText(frame, "DISTRACTION ALERT!", (50, 50), 
                            cv2.FONT_HERSHEY_DUPLEX, 1, (0, 0, 255), 2)
        else:
            phone_start_time = None

        # Hydration logic
        time_since_drink = current_time - last_drink_time
        if time_since_drink > HYDRATION_LIMIT:
            cv2.putText(frame, f"DRINK WATER: {int(time_since_drink)}s", (50, 100), 
                        cv2.FONT_HERSHEY_DUPLEX, 1, (255, 120, 0), 2)
        else:
            cv2.putText(frame, "STATUS: FOCUSED", (50, 50), 
                        cv2.FONT_HERSHEY_DUPLEX, 0.8, (0, 255, 0), 2)

        # Show frame
        cv2.imshow('AuraGuard System', frame)

        # Press 'q' to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    # --- STEP 6: CLEANUP ---
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released and windows closed.")

AuraGuard Active. Focus on the popup window. Press 'q' to stop.
Camera released and windows closed.
